## **SQL Analysis**

**Tujuan**

Melakukan analisis data pembelajaran menggunakan SQL untuk mengidentifikasi distribusi hasil belajar, tingkat aktivitas mahasiswa, dan hubungan antara student engagement dengan learning outcomes.

**Dataset**
- studentInfo
- studentRegistration
- studentVle (agregasi aktivitas mahasiswa)
- courses

**Pertanyaan Bisnis**
1. Bagaimana distribusi hasil akhir pembelajaran?
2. Berapa tingkat kelulusan dan pengunduran diri mahasiswa?
3. Bagaimana tingkat aktivitas belajar berdasarkan hasil akhir?
4. Apakah mahasiswa dengan engagement tinggi cenderung memiliki hasil pembelajaran yang lebih baik?

#### **Menghubungkan Google Drive**

In [ ]:
import pandas as pd
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

base_path = Path("/content/drive/MyDrive/Data_Analytics/edtech-learning-analytics")

processed_path = base_path / "data" / "processed"

print("Processed folder:", processed_path)

Mounted at /content/drive
Processed folder: /content/drive/MyDrive/Data_Analytics/edtech-learning-analytics/data/processed


#### **Membaca Dataset**

In [ ]:
student_info = pd.read_csv(processed_path / "studentInfo_clean.csv")

student_registration = pd.read_csv(processed_path / "studentRegistration_clean.csv")

courses = pd.read_csv(processed_path / "courses_clean.csv")

print("Student Info:", student_info.shape)
print("Student Registration:", student_registration.shape)
print("Courses:", courses.shape)

Student Info: (32593, 12)
Student Registration: (32593, 5)
Courses: (22, 3)


#### **Membuat Agregasi Aktivitas mahasiswa**

In [ ]:
# Memuat Student VLE
student_vle = pd.read_csv(processed_path / "studentVle_clean.csv")

print("Jumlah baris:", len(student_vle))
print(student_vle.head())

Jumlah baris: 10655280
  code_module code_presentation  id_student  id_site  date  sum_click
0         AAA             2013J       28400   546652   -10          4
1         AAA             2013J       28400   546652   -10          1
2         AAA             2013J       28400   546652   -10          1
3         AAA             2013J       28400   546614   -10         11
4         AAA             2013J       28400   546714   -10          1


In [ ]:
# Agregasi Keterlibatan mahasiswa
student_key = ["code_module", "code_presentation", "id_student"]

student_engagement = (
    student_vle
    .groupby(student_key, as_index=False)
    .agg(
        total_clicks=("sum_click", "sum"),
        active_days=("date", "nunique"),
        unique_resources=("id_site", "nunique")
    )
)

print("Jumlah baris hasil agregasi:", len(student_engagement))

student_engagement.head()

Jumlah baris hasil agregasi: 29228


,code_module,code_presentation,id_student,total_clicks,active_days,unique_resources
0,AAA,2013J,11391,934,40,55
1,AAA,2013J,28400,1435,80,84
2,AAA,2013J,30268,281,12,22
3,AAA,2013J,31604,2158,123,82
4,AAA,2013J,32885,1034,70,66


In [ ]:
print("Ukuran dataset:", student_engagement.shape)

Ukuran dataset: (29228, 6)


####**Validasi Hasil Agregasi**

In [ ]:
# Validasi Total Klik
print("Total klik data asli:", student_vle["sum_click"].sum())

print("Total klik hasil agregasi:", student_engagement["total_clicks"].sum())

print("Apakah total klik sama?", student_vle["sum_click"].sum() == student_engagement["total_clicks"].sum())

Total klik data asli: 39605099
Total klik hasil agregasi: 39605099
Apakah total klik sama? True


#### **Menyimpan Dataset Hasil Agregasi**

In [ ]:
student_engagement.to_csv(processed_path / "student_engagement.csv", index=False)

print("Student engagement berhasil disimpan.")

Student engagement berhasil disimpan.


### **Penyiapan Basis Data dan Impor Data (Supabase)**

Database PostgreSQL disiapkan menggunakan Supabase untuk menyimpan dataset hasil preprocessing dan menjalankan analisis SQL.

**Tabel Basis Data**

Dua tabel utama dibuat pada schema public:

- public.student_info
- public.student_engagement

Tabel student_info menyimpan informasi mahasiswa dan hasil akhir pembelajaran, sedangkan tabel student_engagement menyimpan hasil agregasi aktivitas mahasiswa.

**Impor Data**

Dataset diimpor melalui fitur Import Data from CSV pada Supabase Table Editor.

File yang digunakan:
- studentInfo_clean.csv
- student_engagement.csv

Proses impor dilakukan setelah struktur tabel dibuat melalui SQL Editor.

### **Validasi Data dengan SQL**

Validasi dilakukan untuk memastikan jumlah baris yang berhasil diimpor ke PostgreSQL sesuai dengan hasil pengolahan di Google Colab.

**SQL Query**

```sql
SELECT COUNT(*) AS total_records
FROM public.student_info;

SELECT COUNT(*) AS total_records
FROM public.student_engagement;
```

**Hasil**

| Table | Total Records |
|---|---:|
| student_info | 32.593 |
| student_engagement | 29.228 |

**Interpretasi**

Jumlah baris pada kedua tabel sesuai dengan dataset hasil pengolahan di Google Colab. Hal ini menunjukkan bahwa jumlah record yang diimpor telah sesuai dengan data sumber yang digunakan pada tahap ini.

### **Distribusi Hasil Pembelajaran**

**Tujuan**

Menghitung jumlah dan persentase observasi mahasiswa berdasarkan kategori hasil akhir pembelajaran.

**SQL Query**

```sql
SELECT
    final_result,
    COUNT(*) AS total_students,
    ROUND(
        COUNT(*) * 100.0 /
        SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM public.student_info
GROUP BY final_result
ORDER BY total_students DESC;
```

**Hasil**

| Final Result | Total Students | Percentage |
|---|---:|---:|
| Pass | 12.361 | 37,93% |
| Withdrawn | 10.156 | 31,16% |
| Fail | 7.052 | 21,64% |
| Distinction | 3.024 | 9,28% |

**Temuan Awal**

- Pass merupakan kategori hasil akhir terbanyak dengan persentase 37,93%.
- Sebanyak 31,16% observasi berstatus Withdrawn.
- Sebanyak 21,64% observasi berstatus Fail.
- Sebanyak 9,28% observasi memperoleh Distinction.

Tingginya proporsi Withdrawn menunjukkan bahwa pengunduran diri merupakan salah satu aspek penting untuk dianalisis lebih lanjut dalam konteks pembelajaran daring.


### **Analisis Keterlibatan mahasiswa**

**Tujuan**

Menganalisis perbedaan tingkat aktivitas belajar berdasarkan hasil akhir pembelajaran menggunakan SQL INNER JOIN.

**SQL Query**

```sql
SELECT
    si.final_result,
    COUNT(*) AS total_students,
    ROUND(AVG(se.total_clicks), 2) AS avg_total_clicks,
    ROUND(AVG(se.active_days), 2) AS avg_active_days,
    ROUND(AVG(se.unique_resources), 2) AS avg_unique_resources
FROM public.student_info AS si
INNER JOIN public.student_engagement AS se
    ON si.code_module = se.code_module
    AND si.code_presentation = se.code_presentation
    AND si.id_student = se.id_student
GROUP BY si.final_result
ORDER BY avg_total_clicks DESC;
```

**Hasil**

| Final Result | Total Students | Avg. Total Clicks | Avg. Active Days | Avg. Unique Resources |
|---|---:|---:|---:|---:|
| Distinction | 3.024 | 2.666,76 | 110,04 | 102,45 |
| Pass | 12.358 | 1.922,28 | 87,07 | 89,08 |
| Fail | 6.678 | 688,36 | 35,04 | 45,49 |
| Withdrawn | 7.168 | 444,82 | 23,08 | 34,33 |

**Temuan Utama**

1. mahasiswa dengan hasil Distinction memiliki rata-rata aktivitas tertinggi pada seluruh metrik engagement, yaitu total klik, hari aktif, dan sumber belajar yang diakses.

2. mahasiswa dengan hasil Pass juga menunjukkan tingkat engagement yang relatif tinggi dibandingkan mahasiswa berstatus Fail dan Withdrawn.

3. mahasiswa berstatus Withdrawn memiliki rata-rata aktivitas terendah, yaitu 444,82 total klik, 23,08 hari aktif, dan 34,33 sumber belajar.

4. Secara deskriptif, terdapat hubungan positif antara tingkat aktivitas pembelajaran dan hasil akhir yang lebih baik.

**Wawasan Bisnis**

Perbedaan tingkat engagement menunjukkan bahwa aktivitas pembelajaran dapat menjadi salah satu indikator yang relevan untuk memantau keterlibatan mahasiswa.

Institusi pendidikan dapat mempertimbangkan pemantauan aktivitas belajar untuk membantu mengidentifikasi mahasiswa yang membutuhkan dukungan tambahan.

Namun, aktivitas rendah tidak secara otomatis berarti mahasiswa akan gagal atau mengundurkan diri. Analisis lanjutan diperlukan sebelum menggunakan engagement sebagai indikator risiko individual.

**Keterbatasan Analisis**

Analisis hanya mencakup observasi mahasiswa-modul-periode yang memiliki aktivitas VLE tercatat karena menggunakan INNER JOIN.

Selain itu, total aktivitas dapat dipengaruhi oleh lamanya mahasiswa mengikuti pembelajaran. Oleh karena itu, hasil ini menunjukkan hubungan deskriptif, bukan hubungan sebab-akibat.

## **Dokumentasi & Temuan**

**1. Persiapan Data**

Dataset studentVle diagregasi berdasarkan kombinasi code_module, code_presentation, dan id_student.

Hasil agregasi menghasilkan 29.228 baris dan 6 kolom, termasuk tiga metrik aktivitas utama: total_clicks, active_days, dan unique_resources.

Dataset agregasi disimpan sebagai student_engagement.csv.

**2. Penyiapan Basis Data**

Database PostgreSQL disiapkan menggunakan Supabase.

Dua tabel utama yang digunakan:
- student_info
- student_engagement

**3. Validasi Impor Data**

Hasil validasi menunjukkan:
- student_info: 32.593 baris.
- student_engagement: 29.228 baris.

Jumlah baris kedua tabel sesuai dengan hasil pengolahan sebelumnya di Google Colab.

**4. Analisis Hasil Pembelajaran**

Distribusi hasil akhir pembelajaran menunjukkan:
- Pass: 37,93%.
- Withdrawn: 31,16%.
- Fail: 21,64%.
- Distinction: 9,28%.

Proporsi pengunduran diri yang cukup besar menjadi salah satu temuan awal yang relevan untuk analisis lebih lanjut.

**5. Analisis Keterlibatan mahasiswa**

Analisis dilakukan dengan menggabungkan student_info dan student_engagement menggunakan SQL INNER JOIN.

Metrik yang dianalisis mencakup rata-rata total klik, jumlah hari aktif, dan jumlah sumber belajar yang diakses berdasarkan kategori hasil akhir.

**6. Hasil:**
| Final Result | Total Students | Avg. Total Clicks | Avg. Active Days | Avg. Unique Resources |
|---|---:|---:|---:|---:|
| Distinction | 3.024 | 2.666,76 | 110,04 | 102,45 |
| Pass | 12.358 | 1.922,28 | 87,07 | 89,08 |
| Fail | 6.678 | 688,36 | 35,04 | 45,49 |
| Withdrawn | 7.168 | 444,82 | 23,08 | 34,33 |

**7. Keterbatasan Analisis**

- Aktivitas pembelajaran dihitung berdasarkan seluruh catatan studentVle yang dipertahankan pada tahap cleaning.
- Perhitungan engagement menggunakan total klik, jumlah hari aktif, dan jumlah sumber belajar sebagai indikator aktivitas, bukan ukuran langsung pemahaman mahasiswa.
- Analisis menggunakan INNER JOIN hanya mencakup observasi yang memiliki aktivitas VLE tercatat.
- Hubungan antara engagement dan hasil akhir pembelajaran tidak dapat langsung ditafsirkan sebagai hubungan sebab-akibat.